Notebook 1
Ingesta de Datos Reales con yfinance

CELDA 1 — Instalación

In [10]:
!pip install yfinance ta --quiet

CELDA 2 — Importaciones

In [11]:
import yfinance as yf
import pandas as pd
import numpy as np
import ta
import json

CELDA 3 — Configuración

In [12]:
tickers = [
    "FSM",
    "VOLCABC1.LM",
    "ABX.TO",
    "BVN",
    "BHP"
]

periodo = "2y"

CELDA 4 — Descarga de datos

In [13]:
datos = {}

for ticker in tickers:
    try:
        df = yf.download(
            ticker,
            period=periodo,
            auto_adjust=False,
            progress=False
        )

        # Si las columnas vienen multinivel
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        datos[ticker] = df

        print(f"{ticker}: {len(df)} registros")

    except Exception as e:
        print(f"Error en {ticker}: {e}")

FSM: 501 registros
VOLCABC1.LM: 494 registros
ABX.TO: 502 registros
BVN: 501 registros
BHP: 501 registros


CELDA 5 — Indicadores técnicos

In [16]:
for ticker in datos:

    df = datos[ticker]

    # SMA
    df["SMA20"] = ta.trend.sma_indicator(
        df["Close"],
        window=20
    )

    df["SMA50"] = ta.trend.sma_indicator(
        df["Close"],
        window=50
    )

    # EMA
    df["EMA12"] = ta.trend.ema_indicator(
        df["Close"],
        window=12
    )

    df["EMA26"] = ta.trend.ema_indicator(
        df["Close"],
        window=26
    )

    # RSI-14
    df["RSI14"] = ta.momentum.rsi(
        df["Close"],
        window=14
    )

    # MACD
    macd = ta.trend.MACD(df["Close"])

    df["MACD"] = macd.macd()
    df["MACD_SIGNAL"] = macd.macd_signal()

    # Bollinger Bands
    boll = ta.volatility.BollingerBands(
        df["Close"],
        window=20,
        window_dev=2
    )

    df["BB_UPPER"] = boll.bollinger_hband()
    df["BB_LOWER"] = boll.bollinger_lband()

    datos[ticker] = df

print("Indicadores calculados.")

Indicadores calculados.


CELDA 6 — Conversión al JSON

In [17]:
json_salida = {}

for ticker in datos:

    df = datos[ticker].reset_index()

    print(df.columns)   # puedes quitar esta línea después

    json_salida[ticker] = {

        "fechas": df["Date"].astype(str).tolist(),

        "opens": df["Open"].round(4).tolist(),
        "highs": df["High"].round(4).tolist(),
        "lows": df["Low"].round(4).tolist(),
        "closes": df["Close"].round(4).tolist(),

        "volumes": df["Volume"].fillna(0).astype(int).tolist(),

        "sma20": df["SMA20"].replace(np.nan, None).tolist(),
        "sma50": df["SMA50"].replace(np.nan, None).tolist(),

        "ema12": df["EMA12"].replace(np.nan, None).tolist(),
        "ema26": df["EMA26"].replace(np.nan, None).tolist(),

        "rsi14": df["RSI14"].replace(np.nan, None).tolist(),

        "macd": df["MACD"].replace(np.nan, None).tolist(),
        "macd_signal": df["MACD_SIGNAL"].replace(np.nan, None).tolist(),

        "bb_upper": df["BB_UPPER"].replace(np.nan, None).tolist(),
        "bb_lower": df["BB_LOWER"].replace(np.nan, None).tolist()
    }

Index(['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'SMA20',
       'SMA50', 'EMA12', 'EMA26', 'RSI', 'MACD', 'RSI14', 'MACD_SIGNAL',
       'BB_UPPER', 'BB_LOWER'],
      dtype='object', name='Price')
Index(['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'SMA20',
       'SMA50', 'EMA12', 'EMA26', 'RSI', 'MACD', 'RSI14', 'MACD_SIGNAL',
       'BB_UPPER', 'BB_LOWER'],
      dtype='object', name='Price')
Index(['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'SMA20',
       'SMA50', 'EMA12', 'EMA26', 'RSI', 'MACD', 'RSI14', 'MACD_SIGNAL',
       'BB_UPPER', 'BB_LOWER'],
      dtype='object', name='Price')
Index(['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'SMA20',
       'SMA50', 'EMA12', 'EMA26', 'RSI', 'MACD', 'RSI14', 'MACD_SIGNAL',
       'BB_UPPER', 'BB_LOWER'],
      dtype='object', name='Price')
Index(['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'SMA20',
       'SMA50', 'EMA12', 'EMA26', 'RSI', 

CELDA 7 — Exportación

In [18]:
with open("datos_mercado.json", "w") as archivo:
    json.dump(
        json_salida,
        archivo,
        indent=4
    )

print("Archivo datos_mercado.json generado.")

Archivo datos_mercado.json generado.


CELDA 8 — Descarga del JSON

In [19]:
from google.colab import files

files.download("datos_mercado.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>